### Célula 0

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from databricks import sql

load_dotenv(override=True)

connection = sql.connect(
    server_hostname=os.environ["DATABRICKS_HOST"],
    http_path=os.environ["DATABRICKS_HTTP_PATH"],
    access_token=os.environ["DATABRICKS_TOKEN"],
    catalog=os.environ.get("DATABRICKS_CATALOG", "dev"),
    schema="gold",
)

def run_query(query: str) -> pd.DataFrame:
    """Executa uma query no SQL Warehouse e retorna um DataFrame pandas."""
    with connection.cursor() as cursor:
        cursor.execute(query)
        if cursor.description is None:
            return pd.DataFrame()
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        return pd.DataFrame(rows, columns=columns)

### Célula 1

In [2]:
df_mensal = run_query("""
    SELECT DISTINCT product_variant_id, location_id, pct_cobertura
    FROM fct_demanda_mensal_produto_local
""")

pct_zero_mensal = 1 - df_mensal["pct_cobertura"].mean()
print(f"% médio de células (mês) com demanda zero: {pct_zero_mensal:.1%}")

% médio de células (mês) com demanda zero: 73.4%


### Célula 2

In [3]:
run_query("""
CREATE OR REPLACE TEMP VIEW semanal_base AS
SELECT
    f.product_variant_id,
    f.location_id,
    date_trunc('week', c.data) AS semana_ref,
    SUM(f.quantity) AS quantidade_semana
FROM fct_orders f
JOIN dim_calendar c ON f.placed_at_key = c.date_key
GROUP BY f.product_variant_id, f.location_id, date_trunc('week', c.data)
""")

df_semanal = run_query("""
    SELECT
        product_variant_id,
        location_id,
        COUNT(DISTINCT semana_ref) AS semanas_com_venda,
        (datediff(MAX(semana_ref), MIN(semana_ref)) / 7) + 1 AS semanas_de_vida
    FROM semanal_base
    GROUP BY product_variant_id, location_id
""")

df_semanal["pct_cobertura_semanal"] = df_semanal["semanas_com_venda"] / df_semanal["semanas_de_vida"]
pct_zero_semanal = 1 - df_semanal["pct_cobertura_semanal"].mean()
print(f"% médio de células (semana) com demanda zero: {pct_zero_semanal:.1%}")

% médio de células (semana) com demanda zero: 92.9%


### Célula 3

In [4]:
print(f"Grão mensal : {pct_zero_mensal:.1%} das células com demanda zero")
print(f"Grão semanal: {pct_zero_semanal:.1%} das células com demanda zero")
print(f"Diferença   : {pct_zero_semanal - pct_zero_mensal:.1%} pontos percentuais a mais de zeros no semanal")

Grão mensal : 73.4% das células com demanda zero
Grão semanal: 92.9% das células com demanda zero
Diferença   : 19.5% pontos percentuais a mais de zeros no semanal
